# 3.2 Data Preparation

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [9]:
!wget $data -O Churn_pred_data.csv

--2025-11-08 05:57:19--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘Churn_pred_data.csv’

Churn_pred_data.csv 100%[===================>] 954.59K  3.96MB/s    in 0.2s    

2025-11-08 05:57:19 (3.96 MB/s) - ‘Churn_pred_data.csv’ saved [977501/977501]



In [2]:
df = pd.read_csv("Churn_pred_data.csv")

In [3]:
df.columns = df.columns.str.lower().str.replace(" ","_")

catogerical_columns = list(df.dtypes[df.dtypes == 'object'].index)
for i in catogerical_columns:
    df[i] = df[i].str.lower().str.replace(" ","_")

In [4]:
tc = pd.to_numeric(df.totalcharges, errors='coerce')

In [5]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')

In [6]:
df.totalcharges = df.totalcharges.fillna(0)

In [7]:
df.churn

0        no
1        no
2       yes
3        no
4       yes
       ... 
7038     no
7039     no
7040     no
7041    yes
7042     no
Name: churn, Length: 7043, dtype: object

In [8]:
df.churn = (df.churn == 'yes').astype(int)

# 3.3 Setting up validation Framework

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
df_full_train,df_test = train_test_split(df, test_size = 0.2, random_state = 13)

In [11]:
df_train,df_val = train_test_split(df_full_train, test_size = 0.25, random_state = 13)

In [12]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [13]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

In [14]:
del df_train["churn"]
del df_val["churn"]
del df_test["churn"]

# 3.4 EDA

In [15]:
df_full_train = df_full_train.reset_index(drop = True)

In [16]:
df_full_train.churn.value_counts(normalize=True)

churn
0    0.732339
1    0.267661
Name: proportion, dtype: float64

In [17]:
global_churn_rate = df_full_train.churn.mean()
round(global_churn_rate,2)

np.float64(0.27)

In [18]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',
       'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

In [19]:
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

# 3.5 Feature Importance: Churn Rate And Risk Ratio

### Churn Rate

In [20]:
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
print('female: ',churn_female)
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()
print('male: ',churn_male)
print('Global: ',global_churn_rate)

female:  0.2693409742120344
male:  0.2660098522167488
Global:  0.26766063187788425


In [21]:
df_full_train.partner.value_counts()

partner
no     2927
yes    2707
Name: count, dtype: int64

In [22]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_partner

np.float64(0.19837458441078684)

In [23]:
global_churn_rate - churn_partner

np.float64(0.0692860474670974)

In [24]:
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()
churn_no_partner

np.float64(0.33173898189272294)

In [25]:
global_churn_rate - churn_no_partner

np.float64(-0.06407835001483869)

### Risk Ratio

In [26]:
print('risk ratio_no partner: ',churn_no_partner/global_churn_rate)

risk ratio_no partner:  1.2394014747901865


In [27]:
print('risk ratio_yes partner: ',churn_partner/global_churn_rate)

risk ratio_yes partner:  0.7411421807495844


In [28]:
from IPython.display import display

In [29]:
for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean','count'])
    df_group['diff'] = global_churn_rate - df_group['mean']
    df_group['risk'] = df_group['mean']/global_churn_rate
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.269341,2792,-0.001680,1.006278
male,0.266010,2842,0.001651,0.993833




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.237471,4729,0.030190,0.887209
1,0.425414,905,-0.157754,1.589380




partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.427003,3096,-0.159342,1.595313
one_year,0.121471,1169,0.146189,0.453826
two_year,0.032140,1369,0.235520,0.120078




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.165053,2272,0.102608,0.616650
yes,0.337002,3362,-0.069341,1.259064




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.170024,1241,0.097636,0.635223
credit_card_(automatic),0.156536,1201,0.111124,0.584831
electronic_check,0.458511,1880,-0.190850,1.713030
mailed_check,0.188262,1312,0.079398,0.703362


# 3.6 Feature Importance: Mutual Information

In [47]:
from sklearn.metrics import mutual_info_score

In [48]:
mutual_info_score(df_full_train.gender, df_full_train.churn)

7.075416509072507e-06

In [49]:
mutual_info_score(df_full_train.contract, df_full_train.churn)

0.09462729887750038

In [53]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn) 

In [55]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending = False)

contract            0.094627
onlinesecurity      0.062865
techsupport         0.061751
internetservice     0.055779
onlinebackup        0.045619
paymentmethod       0.045218
deviceprotection    0.043274
streamingmovies     0.032173
streamingtv         0.031916
paperlessbilling    0.018944
dependents          0.016038
partner             0.011466
seniorcitizen       0.011273
multiplelines       0.000750
phoneservice        0.000029
gender              0.000007
dtype: float64